# FEM Simulation Convergence Analysis

This notebook visualizes the iteration convergence history for all FEM demo simulations using the custom matplotlib styling template.

## Demos Analyzed:
1. **fem_demo0**: Single hole plate (3170 DOFs, 416 iterations)
2. **fem_demo0_3holes**: Three holes plate (3096 DOFs, 23 iterations)
3. **fem_demo0_3holes_full**: Three holes full mesh (30810 DOFs, 745 iterations)

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Import beaut4mpl v2.0 - matplotlib-style (OOP) API
import sys
sys.path.insert(0, '../..')  # repo root; run from analysis/job_hist_postprocessing/
from beaut4mpl.mplots import LinePlot

# Create a single plotter instance to use throughout
plotter = LinePlot()

# Get color palette for consistent styling
colors = plotter.get_color_palette('Spring_Pastels', n=3)

print("✓ Imports successful")
print(f"✓ Using beaut4mpl v2.0 with colors: {colors}")

## Load FEM Simulation Data

In [ ]:
import os
# Define paths to FEM simulation data
base_path = Path(os.environ.get("OKR_RUNS_ROOT", "../../runs")) / "jobs"

demos = {
    'fem_demo0': {
        'name': 'Single Hole Plate',
        'path': base_path / 'session-atomagent-fem_demo0' / 'analysis',
        'color_idx': 0
    },
    'fem_demo0_3holes': {
        'name': 'Three Holes Plate',
        'path': base_path / 'session-atomagent-fem_demo0_3holes' / 'analysis',
        'color_idx': 1
    },
    'fem_demo0_3holes_full': {
        'name': 'Three Holes Full Mesh',
        'path': base_path / 'session-atomagent-fem_demo0_3holes_full' / 'analysis',
        'color_idx': 2
    }
}

# Load data for each demo
data = {}
for key, demo in demos.items():
    # Load residuals CSV
    residuals_file = demo['path'] / 'residuals.csv'
    metrics_file = demo['path'] / 'convergence_metrics.json'
    
    if residuals_file.exists() and metrics_file.exists():
        df = pd.read_csv(residuals_file)
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
        
        # Remove duplicate rows (every other row is duplicate in some files)
        df_unique = df.drop_duplicates(subset=['iter'], keep='first')
        
        data[key] = {
            'residuals': df_unique,
            'metrics': metrics,
            'name': demo['name'],
            'color_idx': demo['color_idx']
        }
        
        print(f"✓ Loaded {key}: {len(df_unique)} iterations, "
              f"{metrics['ndofs']} DOFs, "
              f"final residual: {metrics['final_residual']:.2e}")
    else:
        print(f"✗ Data not found for {key}")

print(f"\n✓ Successfully loaded {len(data)} FEM simulations")

## 1. Individual Convergence Plots (Linear Scale)

Plot residual vs iteration for each simulation separately.

In [ ]:
# Get color palette
styler = LinePlot()

# Create subplots for each demo
fig, axes = styler.create_figure(2, 2, figsize=(12, 10))
axes = axes.flatten()[:len(data)]  # Adjust axes in case of fewer demos
for idx, (key, demo_data) in enumerate(data.items()):
    ax = axes[idx]
    df = demo_data['residuals']
    metrics = demo_data['metrics']
    
    # Plot residual history
    styler.plot(
        ax, 
        df['iter'], 
        df['residual'], 
        # color=colors[demo_data['color_idx']],
        linewidth=2
    )
    
    # Style the axes
    styler.set_labels(
        ax,
        xlabel="Iteration",
        ylabel="Residual",
        title=f"{demo_data['name']}"
    )
    
    styler.style_axes(ax)

plt.tight_layout()
plt.show()

## 2. Individual Convergence Plots (Log Scale)

Log scale better shows the exponential convergence behavior.

In [ ]:
# Create subplots with log scale
plotter = LinePlot()
fig, axes = plotter.create_figure(1, 3, figsize=(18, 5))

for idx, (key, demo_data) in enumerate(data.items()):
    ax = axes[idx]
    df = demo_data['residuals']
    metrics = demo_data['metrics']
    
    # Plot residual history on log scale
    plotter.plot(
        ax, 
        df['iter'], 
        df['residual'], 
        color=colors[demo_data['color_idx']],
        linewidth=2
    )
    
    # Set log scale
    ax.set_yscale('log')
    
    # Finalize with labels
    
    # Add convergence rate info
    # Calculate approximate convergence rate from last 20% of iterations
    n_late = max(5, len(df) // 5)
    late_iters = df['iter'].iloc[-n_late:]
    late_resid = df['residual'].iloc[-n_late:]
    
    if len(late_iters) > 1:
        # Fit linear in log space
        log_resid = np.log10(late_resid)
        coeffs = np.polyfit(late_iters, log_resid, 1)
        rate = coeffs[0]
        
        ax.text(
            0.95, 0.95,
            f"Final: {metrics['final_residual']:.2e}\nIters: {metrics['iterations']}\nRate: {rate:.3f} log/iter",
            transform=ax.transAxes,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=colors[demo_data['color_idx']]),
            fontsize=11
        )

plt.tight_layout()
plt.show()

### 2.1 Single Hole Plate (fem_demo0) - Detailed View

In [ ]:
# Individual plot for fem_demo0
demo_data = data['fem_demo0']
df = demo_data['residuals']
metrics = demo_data['metrics']

# Create figure
fig, ax = plotter.create_figure(figsize=(8, 6))

# Plot residual history
plotter.plot(
    ax, 
    df['iter'], 
    df['residual'], 
    color=colors[demo_data['color_idx']],
    linewidth=2.5
)

# Set log scale
ax.set_yscale('log')

# Calculate convergence rate from last 20% of iterations
n_late = max(5, len(df) // 5)
late_iters = df['iter'].iloc[-n_late:]
late_resid = df['residual'].iloc[-n_late:]

if len(late_iters) > 1:
    log_resid = np.log10(late_resid)
    coeffs = np.polyfit(late_iters, log_resid, 1)
    rate = coeffs[0]
    
    # Add info box
    info_text = f"Initial residual: {df['residual'].iloc[0]:.2e}\n"
    info_text += f"Final residual: {metrics['final_residual']:.2e}\n"
    info_text += f"Total iterations: {metrics['iterations']}\n"
    info_text += f"Elapsed time: {metrics['elapsed_s']:.3f}s\n"
    info_text += f"Convergence rate: {rate:.3f} log/iter"
    
    ax.text(
        0.97, 0.97,
        info_text,
        transform=ax.transAxes,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, 
                  edgecolor=colors[demo_data['color_idx']], linewidth=2),
        fontsize=14,
        family='monospace'
    )

# Add grid for better readability
ax.grid(True, which='both', alpha=0.3, linestyle='--', linewidth=0.5)

# Finalize the plot
plotter.finalize(
    ax,
    xlabel="Iteration",
    ylabel="Residual (log scale)",
    title=f"{demo_data['name']} Convergence History"
)

plt.tight_layout()
plt.show()

### 2.2 Three Holes Plate (fem_demo0_3holes) - Detailed View

In [ ]:
# Individual plot for fem_demo0_3holes
demo_data = data['fem_demo0_3holes']
df = demo_data['residuals']
metrics = demo_data['metrics']

# Create figure
fig, ax = plotter.create_figure(figsize=(8, 6))

# Plot residual history
plotter.plot(
    ax, 
    df['iter'], 
    df['residual'], 
    color=colors[demo_data['color_idx']],
    linewidth=2.5
)

# Set log scale
ax.set_yscale('log')

# Calculate convergence rate from last 20% of iterations
n_late = max(5, len(df) // 5)
late_iters = df['iter'].iloc[-n_late:]
late_resid = df['residual'].iloc[-n_late:]

if len(late_iters) > 1:
    log_resid = np.log10(late_resid)
    coeffs = np.polyfit(late_iters, log_resid, 1)
    rate = coeffs[0]
    
    # Add info box
    info_text = f"Initial residual: {df['residual'].iloc[0]:.2e}\n"
    info_text += f"Final residual: {metrics['final_residual']:.2e}\n"
    info_text += f"Total iterations: {metrics['iterations']}\n"
    info_text += f"Elapsed time: {metrics['elapsed_s']:.3f}s\n"
    info_text += f"Convergence rate: {rate:.3f} log/iter\n"
    info_text += f"\n✓ Fastest convergence!"
    
    ax.text(
        0.97, 0.97,
        info_text,
        transform=ax.transAxes,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, 
                  edgecolor=colors[demo_data['color_idx']], linewidth=2),
        fontsize=14,
        family='monospace'
    )

# Add grid for better readability
ax.grid(True, which='both', alpha=0.3, linestyle='--', linewidth=0.5)

# Finalize the plot
plotter.finalize(
    ax,
    xlabel="Iteration",
    ylabel="Residual (log scale)",
    title=f"{demo_data['name']} Convergence History"
)

plt.tight_layout()
plt.show()

print(f"\nNote: This simulation converged in only {metrics['iterations']} iterations - the fastest of all three demos!")

### 2.3 Three Holes Full Mesh (fem_demo0_3holes_full) - Detailed View

In [ ]:
# Individual plot for fem_demo0_3holes_full
demo_data = data['fem_demo0_3holes_full']
df = demo_data['residuals']
metrics = demo_data['metrics']

# Create figure
fig, ax = plotter.create_figure(figsize=(10, 8))

# Plot residual history
plotter.plot(
    ax, 
    df['iter'], 
    df['residual'], 
    color=colors[demo_data['color_idx']],
    linewidth=2.5
)

# Set log scale
ax.set_yscale('log')

# Calculate convergence rate from last 20% of iterations
n_late = max(5, len(df) // 5)
late_iters = df['iter'].iloc[-n_late:]
late_resid = df['residual'].iloc[-n_late:]

if len(late_iters) > 1:
    log_resid = np.log10(late_resid)
    coeffs = np.polyfit(late_iters, log_resid, 1)
    rate = coeffs[0]
    
    # Calculate reduction factor
    reduction = df['residual'].iloc[0] / metrics['final_residual']
    
    # Add info box
    info_text = f"Initial residual: {df['residual'].iloc[0]:.2e}\n"
    info_text += f"Final residual: {metrics['final_residual']:.2e}\n"
    info_text += f"Reduction factor: {reduction:.2e}\n"
    info_text += f"Total iterations: {metrics['iterations']}\n"
    info_text += f"Elapsed time: {metrics['elapsed_s']:.3f}s\n"
    info_text += f"Convergence rate: {rate:.3f} log/iter\n"
    info_text += f"\n⚠ Largest problem size"
    
    ax.text(
        0.97, 0.97,
        info_text,
        transform=ax.transAxes,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, 
                  edgecolor=colors[demo_data['color_idx']], linewidth=2),
        fontsize=14,
        family='monospace'
    )

# Add grid for better readability
ax.grid(True, which='both', alpha=0.3, linestyle='--', linewidth=0.5)

# Mark certain iteration milestones
milestones = [100, 200, 300, 400, 500, 600, 700]
for milestone in milestones:
    if milestone < len(df):
        ax.axvline(x=milestone, color='gray', linestyle=':', linewidth=0.8, alpha=0.3)

# Finalize the plot
plotter.finalize(
    ax,
    xlabel="Iteration",
    ylabel="Residual (log scale)",
    title=f"{demo_data['name']} Convergence History"
)

plt.tight_layout()
plt.show()

print(f"\nNote: This is the largest problem with {metrics['ndofs']} DOFs (10x larger than the other demos).")
print(f"Despite the size, it achieved excellent convergence with residual reduction of {reduction:.2e}x")

## 3. Combined Comparison Plot (Log Scale)

All simulations on one plot for direct comparison.

In [ ]:
# Create single plot comparing all demos
fig, ax = plotter.create_figure(figsize=(10, 6))

# Get markers for distinguishing lines
markers = ['o', 's', '^']

# Plot all demos on same axes
for idx, (key, demo_data) in enumerate(data.items()):
    df = demo_data['residuals']
    metrics = demo_data['metrics']
    
    # Plot with markers every Nth point to distinguish lines
    n_markers = min(20, len(df))
    marker_every = max(1, len(df) // n_markers)
    
    ax.plot(
        df['iter'], 
        df['residual'],
        color=colors[demo_data['color_idx']],
        linewidth=2.5,
        marker=markers[idx],
        markersize=6,
        markevery=marker_every,
        label=f"{demo_data['name']} ({metrics['ndofs']} DOFs, {metrics['iterations']} iters)"
    )

# Set log scale
ax.set_yscale('log')

# Add legend
plotter.add_legend(ax, loc='upper right')

# Add grid for better readability on log scale
ax.grid(True, which='both', alpha=0.3, linestyle='--', linewidth=0.5)

# Finalize the plot
plotter.finalize(
    ax,
    xlabel="Iteration",
    ylabel="Residual (log scale)",
    title="FEM Solver Convergence Comparison"
)

plt.tight_layout()
plt.show()

## 4. Normalized Convergence Comparison

Normalize by initial residual to compare convergence rates.

In [ ]:
# Create plot with normalized residuals
fig, ax = plotter.create_figure(figsize=(10, 6))

# Plot normalized convergence
for idx, (key, demo_data) in enumerate(data.items()):
    df = demo_data['residuals']
    metrics = demo_data['metrics']
    
    # Normalize by initial residual
    r0 = df['residual'].iloc[0]
    normalized_residual = df['residual'] / r0
    
    # Plot with markers
    n_markers = min(20, len(df))
    marker_every = max(1, len(df) // n_markers)
    
    ax.plot(
        df['iter'], 
        normalized_residual,
        color=colors[demo_data['color_idx']],
        linewidth=2.5,
        marker=markers[idx],
        markersize=6,
        markevery=marker_every,
        label=f"{demo_data['name']}"
    )

# Set log scale
ax.set_yscale('log')

# Add reference lines for different convergence orders
iters_ref = np.array([0, 50])
for order, label, ls in [(0.1, '0.1/iter', '--'), (0.01, '0.01/iter', ':')]:
    ax.plot(iters_ref, order**iters_ref, 'k', linestyle=ls, linewidth=1, alpha=0.4, label=label)

# Add legend
plotter.add_legend(ax, loc='upper right', ncol=2)

# Add grid
ax.grid(True, which='both', alpha=0.3, linestyle='--', linewidth=0.5)

# Finalize the plot
plotter.finalize(
    ax,
    xlabel="Iteration",
    ylabel="Normalized Residual (r/r₀)",
    title="Normalized FEM Solver Convergence"
)

plt.tight_layout()
plt.show()

## 5. Convergence Metrics Summary

Bar chart comparing key metrics across simulations.

In [ ]:
# Create BarPlot instance for bar charts
from beaut4mpl.mplots import BarPlot
bar_plotter = BarPlot()

# Create subplots for metrics comparison
fig, axes = bar_plotter.create_subplots(1, 3, figsize=(16, 5))

# Prepare data
names = [d['name'] for d in data.values()]
ndofs = [d['metrics']['ndofs'] for d in data.values()]
iters = [d['metrics']['iterations'] for d in data.values()]
elapsed = [d['metrics']['elapsed_s'] for d in data.values()]

x_pos = np.arange(len(names))
hatches_bar = ['///', '\\\\\\', '|||']

# Plot 1: Number of DOFs
for i, (pos, dof, color, hatch) in enumerate(zip(x_pos, ndofs, colors, hatches_bar)):
    bar_plotter.plot(axes[0], [pos], [dof], color=color, hatch=hatch, alpha=0.7)

bar_plotter.finalize(axes[0], xlabel="Simulation", ylabel="Degrees of Freedom", title="Problem Size (DOFs)")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=10)

# Plot 2: Number of Iterations
for i, (pos, iter_count, color, hatch) in enumerate(zip(x_pos, iters, colors, hatches_bar)):
    bar_plotter.plot(axes[1], [pos], [iter_count], color=color, hatch=hatch, alpha=0.7)

bar_plotter.finalize(axes[1], xlabel="Simulation", ylabel="Iterations", title="Iterations to Convergence")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=10)

# Plot 3: Elapsed Time
for i, (pos, time, color, hatch) in enumerate(zip(x_pos, elapsed, colors, hatches_bar)):
    bar_plotter.plot(axes[2], [pos], [time], color=color, hatch=hatch, alpha=0.7)

bar_plotter.finalize(axes[2], xlabel="Simulation", ylabel="Time (seconds)", title="Solver Elapsed Time")
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=10)

plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*70)
print("FEM Simulation Summary")
print("="*70)
print(f"{'Simulation':<25} {'DOFs':>10} {'Iters':>8} {'Time(s)':>10} {'Final Res':>12}")
print("-"*70)
for key, demo_data in data.items():
    m = demo_data['metrics']
    print(f"{demo_data['name']:<25} {m['ndofs']:>10} {m['iterations']:>8} {m['elapsed_s']:>10.3f} {m['final_residual']:>12.2e}")
print("="*70)

## 6. Performance Analysis

Efficiency metrics: iterations per DOF and time per iteration.

In [ ]:
# Create scatter plot: DOFs vs Iterations
from beaut4mpl.mplots import ScatterPlot
scatter_plotter = ScatterPlot()

fig, (ax1, ax2) = scatter_plotter.create_subplots(1, 2, figsize=(14, 6))

# Plot 1: DOFs vs Iterations
for idx, (key, demo_data) in enumerate(data.items()):
    m = demo_data['metrics']
    scatter_plotter.plot(
        ax1,
        [m['ndofs']], 
        [m['iterations']],
        c=colors[demo_data['color_idx']],
        s=300,
        label=demo_data['name'],
        edgecolors='#444444',
        linewidths=2
    )

scatter_plotter.finalize(ax1, xlabel="Degrees of Freedom", ylabel="Iterations", 
                         title="Problem Size vs Convergence Speed")
scatter_plotter.add_legend(ax1, loc='upper left')

# Add text annotations
for idx, (key, demo_data) in enumerate(data.items()):
    m = demo_data['metrics']
    ax1.annotate(
        f"{m['iterations']}",
        (m['ndofs'], m['iterations']),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7, edgecolor='none')
    )

# Plot 2: Time per Iteration
time_per_iter = [d['metrics']['elapsed_s'] / d['metrics']['iterations'] * 1000 
                 for d in data.values()]

x_pos = np.arange(len(names))
for i, (pos, tpi, color, hatch) in enumerate(zip(x_pos, time_per_iter, colors, hatches_bar)):
    bar_plotter.plot(ax2, [pos], [tpi], color=color, hatch=hatch, alpha=0.7)

bar_plotter.finalize(ax2, xlabel="Simulation", ylabel="Time per Iteration (ms)", 
                     title="Solver Performance")
ax2.set_xticks(x_pos)
ax2.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=10)

# Add value labels on bars
for i, (pos, tpi) in enumerate(zip(x_pos, time_per_iter)):
    ax2.text(pos, tpi + max(time_per_iter)*0.02, f'{tpi:.2f}', 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Convergence Rate Analysis

Analyze the local convergence rate throughout the iteration history.

In [ ]:
# Compute and plot local convergence rates
fig, ax = plotter.create_figure(figsize=(10, 6))

for idx, (key, demo_data) in enumerate(data.items()):
    df = demo_data['residuals']
    
    # Compute local convergence rate: r[i+1] / r[i]
    residuals = df['residual'].values
    local_rate = residuals[1:] / residuals[:-1]
    iters = df['iter'].values[:-1]
    
    # Smooth with moving average for clarity
    window = min(10, len(local_rate) // 5)
    if window > 1:
        local_rate_smooth = np.convolve(local_rate, np.ones(window)/window, mode='valid')
        iters_smooth = iters[:len(local_rate_smooth)]
    else:
        local_rate_smooth = local_rate
        iters_smooth = iters
    
    # Plot
    ax.plot(
        iters_smooth, 
        local_rate_smooth,
        color=colors[demo_data['color_idx']],
        linewidth=2,
        label=demo_data['name'],
        alpha=0.8
    )

# Add reference line at rate = 1.0 (no convergence)
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='No convergence')

# Add shaded region for good convergence (rate < 0.1)
ax.axhspan(0, 0.1, alpha=0.1, color='green', label='Fast convergence')

ax.set_ylim(-0.05, 1.2)
plotter.add_legend(ax, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Finalize the plot
plotter.finalize(
    ax,
    xlabel="Iteration",
    ylabel="Local Convergence Rate (r[i+1]/r[i])",
    title="Local Convergence Rate Evolution"
)

plt.tight_layout()
plt.show()

print("\nConvergence Rate Interpretation:")
print("  Rate < 0.5  : Fast linear convergence")
print("  Rate ≈ 0.8-1.0 : Slow convergence")
print("  Rate > 1.0  : Divergence")

## Conclusion

This notebook provides comprehensive visualization of FEM solver convergence behavior across three different simulations:

### Key Findings:
1. **fem_demo0_3holes** converges fastest (23 iterations) despite similar problem size to fem_demo0
2. **fem_demo0_3holes_full** requires most iterations (745) due to 10x larger problem (30,810 DOFs)
3. All solvers achieve excellent convergence (final residual < 1e-11)
4. Time per iteration scales reasonably with problem size

### Style Features Demonstrated:
- **beaut4mpl v2.0** matplotlib-style (OOP) API for publication-ready figures
- Custom color palettes with `Spring_Pastels` (3 colors)
- Fixed font sizes for consistent appearance across all plots
- Log-scale convergence plots for better visualization
- Multi-panel comparisons with consistent styling
- Hatched bar plots for print-friendly figures
- Automatic figure and subplot creation with `LinePlot`, `BarPlot`, and `ScatterPlot` classes